# AI Healthcare Chatbot - Training Triage Text Model
This notebook trains the text classification model using the curated dataset. It replicates the behavior of `backend/scripts/train_text_model.py`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score

print('Libraries imported.')

In [ ]:
DATASET_PATH = 'data/integrated_important_symptom_condition.csv'
df = pd.read_csv(DATASET_PATH)
print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

In [ ]:
# Clean and preprocess text
import re

def clean_text(value: str) -> str:
    text = str(value).strip().lower()
    text = text.replace('|', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text

text_col = next((col for col in ['symptom_text', 'text', 'symptoms'] if col in df.columns), None)
label_col = next((col for col in ['condition', 'label', 'disease'] if col in df.columns), None)

df['clean_text'] = df[text_col].apply(clean_text)
df['target'] = df[label_col].astype(str).str.strip()
print('Text cleaned.')

In [ ]:
x = df['clean_text'].values
y = df['target'].values

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=30000)
X_train_vec = tfidf.fit_transform(x_train)
X_test_vec = tfidf.transform(x_test)
print('TF-IDF vectorization complete.')


In [ ]:
# Filter to keep classes with at least a few samples (e.g. for speed in notebook)
print('Training Logistic Regression model...')
clf = LogisticRegression(max_iter=3000, class_weight='balanced', n_jobs=-1)
clf.fit(X_train_vec, y_train)
print('Training complete.')

In [ ]:
y_pred = clf.predict(X_test_vec)
f1_macro = f1_score(y_test, y_pred, average='macro')
acc = accuracy_score(y_test, y_pred)
print(f"Validation accuracy: {acc:.4f}")
print(f"Validation macro-F1: {f1_macro:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))